# Week 20: Advanced Computer Vision, 3D Vision & NeRFs
## Notebook 03: Neural Radiance Fields (NeRF) & Volumetric Rendering from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Traditionally, 3D objects are modeled using polygon meshes, voxels, or point clouds.
In 2020, Mildenhall et al. (UC Berkeley / Google) revolutionized 3D computer vision with **NeRF (Neural Radiance Fields)**:
- Represents an entire 3D scene inside the weights of an MLP!
- Input: 3D spatial coordinate $(x, y, z)$ and viewing direction $(	heta, phi)$.
- Output: Color $(r, g, b)$ and volume density $sigma$.
- Renders photorealistic novel views using **Classical Volumetric Ray Marching**.

---

### 2. Mathematical Foundations: The Volumetric Rendering Integral
For camera ray $\mathbf{r}(t) = \mathbf{o} + t \mathbf{d}$ cast through 3D space:

$$C(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \cdot \sigma(\mathbf{r}(t)) \cdot \mathbf{c}(\mathbf{r}(t), \mathbf{d}) \, dt$$

Where:
- $\sigma(\mathbf{r}(t))$: Differential volume density (opacity) at point $t$.
- $\mathbf{c}(\mathbf{r}(t), \mathbf{d})$: Emitted RGB color radiance.
- $T(t)$: **Accumulated Transmittance** (probability that the ray travels from $t_n$ to $t$ without hitting any particles):
  $$T(t) = \exp\left( -\int_{t_n}^t \sigma(s) \, ds \right)$$

#### Numerical Quadrature Approximation (Ray Marching):
Discretizing into $N$ sampled intervals along the ray:
$$\hat{C}(\mathbf{r}) = \sum_{i=1}^N T_i \cdot (1 - \exp(-\sigma_i \delta_i)) \cdot \mathbf{c}_i, \quad T_i = \prod_{j=1}^{i-1} \exp(-\sigma_j \delta_j)$$

In [ ]:
import numpy as np

# Step 1: Simulate Volumetric Ray Marching along a Single Camera Ray
# Camera Ray: origin o = [0, 0, -2], direction d = [0, 0, 1] (marching along Z-axis)
num_samples = 8
t_near = 0.5
t_far = 4.0

t_vals = np.linspace(t_near, t_far, num_samples)
deltas = np.diff(t_vals, append=t_vals[-1] - t_vals[-2])

# Step 2: Simulated NeRF MLP Output along the ray
# Suppose a semi-transparent colored sphere exists between t=1.8 and t=2.6
densities = np.array([0.01, 0.02, 0.05, 2.5, 4.0, 1.8, 0.05, 0.01]) # High sigma inside object
colors = np.array([
    [0.1, 0.1, 0.1],
    [0.1, 0.1, 0.1],
    [0.2, 0.2, 0.2],
    [0.9, 0.1, 0.1], # Bright Red Sphere
    [0.9, 0.1, 0.1],
    [0.8, 0.2, 0.1],
    [0.2, 0.2, 0.2],
    [0.1, 0.1, 0.1]
])

# Step 3: Implement Volumetric Quadrature Numerical Integration
def render_ray(densities, colors, deltas):
    # Alpha compositing weight: 1 - exp(-sigma * delta)
    alphas = 1.0 - np.exp(-densities * deltas)
    
    # Accumulated Transmittance T_i = prod_{j=1}^{i-1} (1 - alpha_j)
    transmittance = np.cumprod(np.concatenate([[1.0], 1.0 - alphas[:-1]]))
    
    # Ray weights: w_i = T_i * alpha_i
    weights = transmittance * alphas
    
    # Accumulated RGB Color: sum (w_i * c_i)
    rendered_color = np.sum(weights[:, np.newaxis] * colors, axis=0)
    
    return rendered_color, weights, transmittance

final_rgb, weights, transmittance = render_ray(densities, colors, deltas)

print("--- Volumetric Ray Marching Output ---")
print(f"Rendered Pixel RGB Color: {np.round(final_rgb, 3)}")
print("\nSample Depth Details:")
for i, (t, w, T) in enumerate(zip(t_vals, weights, transmittance)):
    print(f"  Step {i}: Depth t={t:.2f} | Transmittance T={T:.3f} | Sample Weight w={w:.3f}")

### 3. Step-by-Step Code Walkthrough
1. **Accumulated Transmittance:** $T$ starts at 1.0 (empty air) and drops sharply as the ray penetrates the dense object at depth $t=2.0$.
2. **Ray Weighting:** Weights $w_i = T_i \alpha_i$ peak precisely where density is high, integrating the red color into the camera pixel.
3. **Occlusion Handling:** Samples behind the object receive $T \approx 0$, naturally handling 3D occlusion without explicit depth sorting buffers.

---

### 4. Industry Applications & Trade-offs
- **Virtual Reality & VFX:** Generating photorealistic novel viewpoints of real-world environments from smartphone photos.
- **Trade-off:** Training and rendering standard NeRFs requires evaluating millions of MLP forward passes, taking minutes per single frame.

---

### 5. Top Interview Questions & Answers
**Q1: Why is Positional Encoding $(\gamma(x))$ necessary in NeRF?**
> **Answer:** Rahaman et al. proved that standard coordinate MLPs suffer from spectral bias: they only learn low-frequency functions, producing blurry renders. Mapping coordinates to high-frequency sinusoids $\gamma(p) = (\sin(2^0 \pi p), \cos(2^0 \pi p), \dots, \sin(2^{L-1} \pi p), \cos(2^{L-1} \pi p))$ enables the network to represent sharp edges and fine textures.

**Q2: What is the difference between NeRF and 3D Gaussian Splatting?**
> **Answer:** NeRF represents scenes implicitly via continuous neural network weights and uses slow ray marching. 3D Gaussian Splatting represents scenes explicitly using millions of 3D Gaussians (ellipsoids) and uses tile-based rasterization, achieving real-time 100+ FPS rendering.